# 05 - Baseline Model: Logistic Regression

**Goal:** train a simple, interpretable baseline and get a first honest look
at how hard this prediction problem is.

### Why Logistic Regression first?

- It is a **linear** model - each feature pushes the rain probability up or
  down, and we can see *how much*.
- It trains in seconds.
- It gives us a **probability** output (perfect for our "78% chance of rain"
  style answer).
- Any more complex model must beat this baseline to be worth it.

### Key tools we meet here

| Tool | What it does |
|------|--------------|
| `ColumnTransformer` | applies different transformations to different columns |
| `StandardScaler` | centers/scales numeric columns (mean 0, std 1) |
| `OneHotEncoder` | turns categories (Season) into 0/1 columns |
| `Pipeline` | chains preprocessing + model into one object |

**Critical rule:** the preprocessing must be *fitted on the training data
only*. If we fit the scaler on train+test together, the test set leaks into
training. A `Pipeline` enforces this automatically because it is fitted once
on `X_train`.


## 1. Load the splits from Phase 4


In [1]:
import pandas as pd
import numpy as np

from pathlib import Path
import joblib

SPLITS = Path("data/processed/splits")
if not SPLITS.exists():
    SPLITS = Path("..") / "data/processed/splits"

X_train = pd.read_csv(SPLITS / "X_train.csv")
X_test  = pd.read_csv(SPLITS / "X_test.csv")
y_train = pd.read_csv(SPLITS / "y_train.csv").iloc[:, 0]
y_test  = pd.read_csv(SPLITS / "y_test.csv").iloc[:, 0]

print("X_train:", X_train.shape, "| X_test:", X_test.shape)
print("Rain in train: {:.1f}% | Rain in test: {:.1f}%".format(
    y_train.mean() * 100, y_test.mean() * 100))

X_train: (12156, 16) | X_test: (3040, 16)
Rain in train: 10.4% | Rain in test: 24.8%


## 2. Split columns into numeric and categorical

- **Numeric:** weather measurements, lags, calendar numbers. These need
  scaling (logistic regression is sensitive to scale - a feature in km/h and
  one in % would otherwise be treated with different importance).
- **Categorical:** `Season` (Winter / HotDry / Monsoon / PostMonsoon). This
  needs one-hot encoding (4 binary columns).

> Note: `Month` and `DayOfYear` are treated as numeric here for simplicity.
> Month is really cyclic, but for a baseline this is acceptable.


In [2]:
numeric_features = [
    "MaxTemperature", "MinTemperature", "MeanTemperature",
    "Pressure", "Humidity", "CloudCoverage",
    "WindSpeed", "WindDirection", "Rainfall", "WeatherCode",
    "PreviousRainfall", "PreviousDayTemperature", "PreviousDayHumidity",
    "Month", "DayOfYear",
]
categorical_features = ["Season"]

print("Numeric columns    :", len(numeric_features))
print("Categorical columns:", categorical_features)
assert set(numeric_features + categorical_features) == set(X_train.columns)

Numeric columns    : 15
Categorical columns: ['Season']


## 3. Build the preprocessing pipeline

We compose a `ColumnTransformer`:
- numeric columns -> `StandardScaler()`
- categorical columns -> `OneHotEncoder()`

and wrap it together with the classifier in one `Pipeline`:

```
pipeline = Pipeline([
    ("preprocessor", ColumnTransformer([...])),
    ("classifier",   LogisticRegression(...)),
])
```

One `fit(X_train, y_train)` trains the scaler, the encoder AND the model -
in the correct order, with no leakage.


In [3]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression

preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numeric_features),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features),
    ]
)

model = LogisticRegression(max_iter=1000, random_state=42)

pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", model),
])

print(pipeline)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num', StandardScaler(),
                                                  ['MaxTemperature',
                                                   'MinTemperature',
                                                   'MeanTemperature',
                                                   'Pressure', 'Humidity',
                                                   'CloudCoverage', 'WindSpeed',
                                                   'WindDirection', 'Rainfall',
                                                   'WeatherCode',
                                                   'PreviousRainfall',
                                                   'PreviousDayTemperature',
                                                   'PreviousDayHumidity',
                                                   'Month', 'DayOfYear']),
                                                 ('cat',
                            

## 4. Train the pipeline (fit only on X_train)

This one call:
1. Fits the `StandardScaler` on the **training** data.
2. Fits the `OneHotEncoder` on the **training** data.
3. Trains the Logistic Regression on the scaled/encoded training data.


In [4]:
pipeline.fit(X_train, y_train)
print("Training complete.")

Training complete.


## 5. First evaluation pass

We compute a few metrics for a first look. Deep evaluation (confusion matrix,
ROC curve, threshold analysis) is Phase 6.


In [5]:
from sklearn.metrics import (accuracy_score, precision_score,
                             recall_score, f1_score, roc_auc_score)

y_pred = pipeline.predict(X_test)
y_prob = pipeline.predict_proba(X_test)[:, 1]  # probability of "Rain"

metrics = {
    "Accuracy":  accuracy_score(y_test, y_pred),
    "Precision": precision_score(y_test, y_pred),
    "Recall":    recall_score(y_test, y_pred),
    "F1":        f1_score(y_test, y_pred),
    "ROC-AUC":   roc_auc_score(y_test, y_prob),
}
for name, value in metrics.items():
    print(f"{name:10s}: {value:.3f}")

Accuracy  : 0.848
Precision : 0.828
Recall    : 0.491
F1        : 0.616
ROC-AUC   : 0.917


### First impressions (honest reading)

- **Accuracy** looks high (~80%+) but remember: 75% of *test* days are
  no-rain. A model that always said "No Rain" would score ~75% accuracy for
  free. So accuracy is NOT the story here.
- **Recall** is likely much lower - the model misses many rainy days. This is
  exactly the weakness the next phases address (class imbalance, better
  models, threshold tuning).

We now understand the baseline. Phase 6 evaluates it rigorously.


## 6. Save the baseline model

We save the **whole pipeline** (preprocessing + model) with joblib. Saving the
pipeline - not just the model - guarantees predictions use the exact same
transformations as training.


In [6]:
MODEL_DIR = Path("models")
if not MODEL_DIR.exists():
    MODEL_DIR = Path("..") / "models"
MODEL_DIR.mkdir(parents=True, exist_ok=True)

baseline_path = MODEL_DIR / "karachi_rain_baseline.pkl"
joblib.dump(pipeline, baseline_path)
print("Saved baseline pipeline ->", baseline_path)

Saved baseline pipeline -> ..\models\karachi_rain_baseline.pkl


## Summary of Phase 5

- Built a `Pipeline`: `StandardScaler` for numerics, `OneHotEncoder` for
  Season, then `LogisticRegression`.
- Fitted preprocessing + model **only on training data** (no leakage).
- Got a first, honest baseline with the full metric set to compare against.

**Next phase:** Phase 6 - deep evaluation (confusion matrix, ROC curve, and
why recall matters for rain warnings).
